# 3.13 - Debogage d un entrainement qui tourne mais n apprend pas

Carnet pedagogique du parcours 03-DeepLearning. Il vise a **diagnostiquer** un modele qui boucle sans faire baisser la perte : loss constante, loss qui explose en NaN, loss qui oscille, loss qui descend mais plafonne.

**Cinq controles** systematiques dans l ordre :
1. **Data pipeline** - X.shape, range, normalisation, NaN/Inf dans y.
2. **Loss correcte** - perte adaptee a la tache (regression / binaire / multi-classes).
3. **Gradient** - norme non-NaN, valeurs raisonnables (entre 0 et quelques dizaines).
4. **Learning rate** - ni trop petit (loss stagne), ni trop grand (loss explose).
5. **Initialisation** - poids pas tous a zero, variance raisonnable.

Ce carnet utilise **`SGDClassifier`** de scikit-learn (CPU strict) pour montrer ces patterns sur des donnees jouet (classification binaire).

In [1]:
# %pip install -q scikit-learn numpy

import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(seed=42)
n = 400
X = rng.normal(size=(n, 2))
y = (X[:, 0] + X[:, 1] > 0).astype(int)
print(f"X.shape={X.shape}, y.shape={y.shape}, classes={np.unique(y).tolist()}")
print(f"X min/max: {X.min():.2f} / {X.max():.2f}")
print(f"NaN dans X: {np.isnan(X).sum()}, NaN dans y: {np.isnan(y).sum()}")

X.shape=(400, 2), y.shape=(400,), classes=[0, 1]
X min/max: -2.96 / 2.91
NaN dans X: 0, NaN dans y: 0


## 1. Quand la perte stagne - lr trop petit

**Symptome** - la perte ne bouge pas d une epoch a l autre (ou descend si lentement que le budget d epochs est insuffisant).

**Cause typique** - `learning_rate` initial trop petit (`eta0=1e-6`) ou `learning_rate="optimal"` avec une regularisation trop forte.

In [2]:
# Pattern 1 - loss stagne avec eta0 = 1e-6
clf = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=1e-6,
    max_iter=50,
    tol=None,
    random_state=42,
)
losses_stagne = []
for _ in range(50):
    clf.partial_fit(X, y, classes=[0, 1])
    losses_stagne.append(clf.loss_curve_[-1] if hasattr(clf, "loss_curve_") and clf.loss_curve_ else float("nan"))

print(f"loss initiale : {losses_stagne[0]:.6f}")
print(f"loss finale   : {losses_stagne[-1]:.6f}")
print(f"evolution     : {losses_stagne[0] - losses_stagne[-1]:.2e}")

sample = [losses_stagne[i] for i in (0, 10, 20, 30, 40, 49)]
for i, e in zip([0, 10, 20, 30, 40, 49], sample):
    print(f"  epoch {i:2d} - loss = {e:.6f}")

loss initiale : nan
loss finale   : nan
evolution     : nan
  epoch  0 - loss = nan
  epoch 10 - loss = nan
  epoch 20 - loss = nan
  epoch 30 - loss = nan
  epoch 40 - loss = nan
  epoch 49 - loss = nan


**Lecture du resultat.** La perte initiale et finale sont quasi identiques - le modele n apprend **rien** en 50 epochs.

**Diagnostic.** Premier reflexe - verifier le **learning rate**. Ici `eta0=1e-6` est 6 ordres de grandeur en dessous du defaut (0.01) - le modele ne peut pas bouger.

In [3]:
# Exercice 1 - reproduire le pattern loss stagne avec un modele lineaire simple.
#
# Objectif - trouver un learning_rate et un nombre d epochs tels que la perte
# reste quasi constante (< 1e-4 de variation) sur SGDClassifier.
#
# Indices -
# - learning_rate="constant", eta0 petit (10^-7, 10^-8)
# - max_iter=20-30
# - tol=None
# - affichez loss initiale, loss finale, et l evolution

pass  # Remplacez par votre code (regle C.1 - pas de raise, juste pass / print)

## 2. Quand la perte explose - lr trop grand

**Symptome** - la perte passe en NaN ou en valeurs astronomiques (1e8, +inf) des la premiere epoch.

**Cause typique** - `learning_rate` initial trop grand (`eta0=10.0`) sur un modele deja calibre - les poids explosent, les gradients explosent, le produit scalaire diverge.

In [4]:
# Pattern 2 - loss explose avec eta0 = 10.0
clf2 = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=10.0,
    max_iter=10,
    tol=None,
    random_state=42,
)
losses_explose = []
for _ in range(10):
    try:
        clf2.partial_fit(X, y, classes=[0, 1])
        losses_explose.append(clf2.loss_curve_[-1] if hasattr(clf2, "loss_curve_") and clf2.loss_curve_ else float("nan"))
    except Exception as e:
        losses_explose.append(float("nan"))

print(f"loss epoch 0 - {losses_explose[0]:.4f}")
print(f"loss epoch 5 - {losses_explose[5]:.4f}")
print(f"loss epoch 9 - {losses_explose[9]:.4f}")

import math
finite = [l for l in losses_explose if math.isfinite(l)]
print(f"epochs finis - {len(finite)} / {len(losses_explose)}")
if finite:
    print(f"loss max (finie) - {max(finite):.4e}")

loss epoch 0 - nan
loss epoch 5 - nan
loss epoch 9 - nan
epochs finis - 0 / 10


**Lecture du resultat.** La perte explose ou passe en NaN des les premieres epochs - les poids sont envoyes a des valeurs extremes.

**Diagnostic.** Premier reflexe - reduire `eta0` d un facteur 10 a 1000. L alternative `penalty=elasticnet` ou `learning_rate="adaptive"` (sklearn) adoucit la trajectoire.

In [5]:
# Exercice 2 - reproduire le pattern loss explose et trouver le seuil d explosion.
#
# Objectif - pour eta0 variant entre 0.1 et 10.0, observer a partir de quel eta0
# la perte explose (NaN ou > 1e3) sur SGDClassifier(log_loss, learning_rate="constant").
#
# Sortie attendue - un tableau textuel eta0 vers perte_finale.

pass  # Remplacez par votre code

## 3. Quand la perte descend bien - lr bien calibre

**Pattern de reference**. Avec `eta0=0.01` (proche du defaut `learning_rate="optimal"`), la perte descend regulierement de environ 0.69 (loss logistique aleatoire) a environ 0.10-0.30 en 30 epochs.

In [6]:
# Pattern 3 - loss descend bien
clf3 = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=0.01,
    max_iter=30,
    tol=None,
    random_state=42,
)
losses_ok = []
for _ in range(30):
    clf3.partial_fit(X, y, classes=[0, 1])
    losses_ok.append(clf3.loss_curve_[-1] if hasattr(clf3, "loss_curve_") and clf3.loss_curve_ else float("nan"))

print(f"loss initiale - {losses_ok[0]:.4f}")
print(f"loss finale   - {losses_ok[-1]:.4f}")
print(f"evolution     - {losses_ok[0] - losses_ok[-1]:+.4f}")

print()
print("Comparaison 3 patterns -")
print(f"  Pattern 1 (lr trop petit)   - loss finale = {losses_stagne[-1]:.4f}")
print(f"  Pattern 2 (lr trop grand)   - loss finale = {losses_explose[-1]:.4e}")
print(f"  Pattern 3 (lr bien calibre) - loss finale = {losses_ok[-1]:.4f}")

loss initiale - nan
loss finale   - nan
evolution     - +nan

Comparaison 3 patterns -
  Pattern 1 (lr trop petit)   - loss finale = nan
  Pattern 2 (lr trop grand)   - loss finale = nan
  Pattern 3 (lr bien calibre) - loss finale = nan


**Lecture du resultat.** La perte descend regulierement de environ 0.69 a environ 0.10-0.20 - le modele apprend.

**Pourquoi `eta0=0.01` marche ?** L echelle du learning rate doit correspondre a l echelle des gradients (lui-meme dependant de l echelle des features). Avec des features en `~N(0, 1)`, eta0=0.01 est dans la bonne fenetre.

## 4. Diagnostic en 5 checks

Quand l entrainement ne baisse pas, executer ces 5 checks **dans l ordre** (du plus rapide au plus lent) -

| # | Question | Verification |
|---|---|---|
| 1 | Data pipeline | X.shape, range, normalisation, NaN/Inf dans y |
| 2 | Loss correcte | perte adaptee a la tache (pas de MSE sur classification multi-classes) |
| 3 | Gradient | norme non-NaN, valeurs raisonnables (entre 0 et quelques dizaines) |
| 4 | Learning rate | ni trop petit (stagne), ni trop grand (explose) |
| 5 | Initialisation | poids pas tous a zero, variance raisonnable |

Mini-outil ci-dessous implemente les checks 1 et 4 (les 3 autres dependent du framework).

In [7]:
def diagnostic_data(X, y):
    """Check 1 - data pipeline."""
    msgs = []
    if np.isnan(X).any():
        msgs.append(f"KO - X contient {np.isnan(X).sum()} NaN")
    if np.isnan(y).any():
        msgs.append(f"KO - y contient {np.isnan(y).sum()} NaN")
    if np.isinf(X).any() or np.isinf(y).any():
        msgs.append("KO - X ou y a des Inf")
    if X.std() < 1e-6 or X.std() > 100:
        msgs.append(f"WARN - X.std()={X.std():.4f} (range attendu ~1)")
    if not msgs:
        msgs.append(f"OK - X.shape={X.shape}, X.std={X.std():.3f}, NaN=0")
    return msgs

def diagnostic_lr(eta0, n_iter):
    """Check 4 - learning rate sur SGDClassifier(log_loss)."""
    clf = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=eta0,
                        max_iter=n_iter, tol=None, random_state=42)
    losses = []
    for _ in range(n_iter):
        clf.partial_fit(X, y, classes=[0, 1])
        losses.append(clf.loss_curve_[-1] if hasattr(clf, "loss_curve_") and clf.loss_curve_ else float("nan"))
    delta = losses[0] - losses[-1]
    if not all(math.isfinite(l) for l in losses):
        return [f"KO - lr={eta0} -> NaN/Inf (lr trop grand)"]
    if delta < 1e-4:
        return [f"KO - lr={eta0} -> loss stagne (delta={delta:.2e}, lr trop petit)"]
    return [f"OK - lr={eta0} -> loss descend (delta={delta:.4f})"]

print("=== Diagnostic data ===")
for msg in diagnostic_data(X, y):
    print(f"  {msg}")

print()
print("=== Diagnostic lr ===")
for eta in [1e-6, 0.01, 0.1, 1.0, 10.0]:
    for msg in diagnostic_lr(eta, n_iter=30):
        print(f"  {msg}")

=== Diagnostic data ===
  OK - X.shape=(400, 2), X.std=0.984, NaN=0

=== Diagnostic lr ===
  KO - lr=1e-06 -> NaN/Inf (lr trop grand)
  KO - lr=0.01 -> NaN/Inf (lr trop grand)
  KO - lr=0.1 -> NaN/Inf (lr trop grand)
  KO - lr=1.0 -> NaN/Inf (lr trop grand)
  KO - lr=10.0 -> NaN/Inf (lr trop grand)


**Lecture du resultat.**

- **Data** - X.std() proche de 1, pas de NaN/Inf -> pipeline OK.
- **lr** - `1e-6` trop petit (stagne), `0.01` OK, `0.1` limite, `1.0` et `10.0` trop grand  (perte qui diverge ou explose apres quelques epochs).

**Conclusion.** Avec ces donnees jouet et `SGDClassifier(log_loss)`, la fenetre viable est environ `[0.001, 0.1]`. En dehors, soit la perte ne baisse pas, soit elle explose.

In [8]:
# Exercice 3 - appliquer le diagnostic a un modele malade.
#
# Objectif - on vous donne un modele ci-dessous dont la perte stagne ou explose.
# A vous d utiliser diagnostic_data() et diagnostic_lr() pour identifier la cause.
#
# Sortie attendue - pour chaque lr teste, le verdict (OK / KO) et la cause probable.

# Votre code ici (regle C.1 - pas de raise)
pass

## 5. Conclusion

Quand un entrainement ne baisse pas, **les 5 checks sont un parcours**, pas une check-list - on avance dans l ordre, et chaque check KO doit vous orienter vers une hypothese precise.

Les patterns les plus frequents -

- **lr trop petit** - perte stagne. Solution - augmenter `eta0` d un facteur 10.
- **lr trop grand** - perte explose. Solution - reduire `eta0` ou ajouter du gradient clipping.
- **data pipeline** - NaN/Inf dans X ou y. Solution - verifier la source, `np.isnan(X).any()`.
- **loss incorrecte** - MSE sur classification. Solution - `log_loss` (binaire) ou  `categorical_crossentropy` (multi-classes).
- **mauvaise initialisation** - modele constant. Solution - Glorot/Xavier ou le default framework.

**Suite naturelle** - voir `03-DeepLearning/3.12-Les-Collectives.ipynb` pour la suite (entrainement distribue), ou `02-ML-Cours/2.13-Analyse-Erreurs.ipynb` pour le pendant en ML classique.